In [2]:
import warnings
warnings.filterwarnings('ignore')
#############################################
#BASIC PACKAGES
import sys,time,os
from copy import deepcopy as dcopy
import healpy as hp
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline

#############################################
#PATH TO SCRIPTS FOLDER
path = '/data/AMARINS/LSSxHI-CODES/scripts/'
sys.path.insert(1, path)
import cross_functions_theory      as cxft
import cross_functions_simulations as cxfs
# it's required healpy  (https://healpy.readthedocs.io/en/latest/index.html)
import handling_data               as hdata
import pandas as pd
#############################################
## MATPLOTLIB FEATURES
from matplotlib import cm
font = {'weight' : 'bold','size'   : 22}
mpl.rc('font', **font)
mpl.rc('font',   size=209)  #set defaults so that the plots are readable
mpl.rc('axes',   titlesize=20)
mpl.rc('axes',   labelsize=20)
mpl.rc('xtick',  labelsize=20)
mpl.rc('ytick',  labelsize=20)
mpl.rc('legend', fontsize =20)
mpl.rc('figure', titlesize=20)
mpl.rc('text',   usetex=True)
## FURTHER MATPLOTLIB FEATURES
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.markers import MarkerStyle
from matplotlib.transforms import Affine2D
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
#colors = ["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494"]
colors = ['#ff0000', "#41b6c4", '#0000ff']
my_cmap = ListedColormap(colors, name="my_cmap")
#my_cmap_r = my_cmap.reversed()
#############################################
## NAMASTER (https://namaster.readthedocs.io/en/latest/)
import pymaster as nmt
del_l  = 30
nside  = 256
##############################
b    = nmt.NmtBin.from_nside_linear(nside, nlb=del_l)
leff = b.get_effective_ells()
feff = leff*(leff+1)/2/np.pi


sys.path.insert(1, '/data/AMARINS/chisel/scripts')    
import Extension4BINGO   as cs
import load_standard_params as loadparams
#import pyMRS as pymrs
import statcosmo as statc

In [3]:
def tf_processing(dict_all_=None, reali_mocks=None, reali_obs=None, 
                  params_sims_=None, params_maps_=None, params_inout_=None,
                  tf_estimator=None,
                  params_WT_=None,  params_CS_=None, obs_include_hi=True,
                  verbose=True, verbose_level=0,
                  type_='switzer'):
    f= 1 if obs_include_hi else 0
    job_verbose = verbose==True and verbose_level>0
    cs_verbose  = verbose==True and verbose_level>1
    #reali_mocks=params_sims_['realizations'][1:]
    #reali_obs  =params_sims_['realizations'][0]    
    #FG obs-----        
    Xfg0 = hdata.getmap(dirpath_=params_inout_.dirpath_foregrounds, 
                       filename_=params_inout_.filename_foregrounds, 
                       healpix_readingformat=False, 
                       hdu=1)
    #HI obs-----    
    jsim0      = reali_obs#params_sims_['realizations'][0]
    alm_hi_obs = dict_all_[jsim0]['alm_hi_sim'][2:]
    npix       = hp.nside2npix(params_sims_.nside)
    Xhi0  = np.vstack([ hp.alm2map(np.ascontiguousarray(alm_hi_obs[num]), params_sims_.nside, pol=False, verbose=False) \
                        for num in range(alm_hi_obs.shape[0]) ])        
    #TF-----          
    npix = hp.nside2npix(params_sims_.nside)
    if type_.lower()=='switzer':
        alpha_cx_list = []
        for j, jsim in enumerate(reali_mocks):
            if job_verbose: print('Job {} -- Sim: {}'.format(j+1, jsim))
            params_maps_["getdata"] = "observed"                
            alm_hi_jsim = dict_all_[jsim]['alm_hi_sim'][2:]#mock
            # --- Loading Xhi realization
            Mhi  = np.vstack([ hp.alm2map(np.ascontiguousarray(alm_hi_jsim[num]), params_sims_.nside, pol=False, verbose=False) \
                            for num in range(alm_hi_jsim.shape[0]) ])                
            # --- Filtering 1
            Xfg  = dcopy(Xfg0)
            Xhi  = dcopy(Xhi0)
            WN   = np.zeros_like(Mhi)   
            Xsky = Xfg + f*Xhi;
            params_cs, params_wt = cs.load(params_CS_, params_WT_)
            X  = Xsky + Mhi
            #
            X  = cs.adaptation_maps(X, params_maps_, params_inout_)
            Xr = cs.maps2CSmaps(X, params_wt, params_cs, verbose=cs_verbose)# , False)
            C1 = Xr['reconstruction']['21cm']
            # --- Filtering 2
            Xfg  = dcopy(Xfg0)
            Xhi  = dcopy(Xhi0)
            WN   = np.zeros_like(Mhi)   
            Xsky = Xfg + f*Xhi;
            params_cs, params_wt = cs.load(params_CS_, params_WT_)
            X  = Xsky + 0*Mhi
            #
            X   = cs.adaptation_maps(X, params_maps_, params_inout_)
            Xr  = cs.maps2CSmaps(X, params_wt, params_cs, verbose=cs_verbose)# , False)
            C0  = Xr['reconstruction']['21cm']
            # -------------------------------------
            Xc = C1-C0
            alm_c_sim    = np.vstack([ hp.map2alm(maps=m, pol=False) for m in Xc   ])
            alm_mock_sim = np.vstack([ hp.map2alm(maps=m, pol=False) for m in Mhi ])
            alm_g_sim    = dict_all_[jsim]['alm_kappa_sim'][2:,:]#mock
            #alm_hi_est = np.vstack([ hp.map2alm(maps=m, pol=False) for m in Xr['reconstruction']['21cm'] ])    
            #
            cx_c    = np.vstack([ hp.alm2cl(alms1=alm_g_sim, alms2=i_alm) for i_alm in alm_c_sim    ])
            cx_mock = np.vstack([ hp.alm2cl(alms1=alm_g_sim, alms2=i_alm) for i_alm in alm_mock_sim ])
                
            alpha_cx = cx_mock/cx_c
            alpha_cx_list.append(alpha_cx.flatten())
                
            ndim_cx = alpha_cx.shape
            alpha_cx_flat = np.vstack(alpha_cx_list)
            alpha_cx_sim, alpha_cx_sim_err = statc.get_estimator(alpha_cx_flat, tf_estimator, axis=0)#, **kwargs)
            alpha_cx_sim_err = np.sqrt(alpha_cx_sim_err).reshape(N,-1)
            alpha_cx_sim = alpha_cx_sim.reshape(N,-1)
        return {'tf':alpha_cx_sim, 'errtf':alpha_cx_sim_err, 'realization_mocks': reali_mocks, 'realization_obs':reali_obs,
                'HIobs':Xhi0,'FGobs':Xfg0}           
        
    elif type_.lower()=='fba':
        alpha_cx_list = []
        for j, jsim in enumerate(reali_mocks):
            if job_verbose: print('Job {} -- Sim: {}'.format(j+1, jsim))
            params_maps_["getdata"] = "observed"                
            alm_hi_jsim = dict_all_[jsim]['alm_hi_sim'][2:]#mock
            # --- Loading Xhi realization
            npix = hp.nside2npix(params_sims_.nside)
            Mhi  = np.zeros((alm_hi_jsim.shape[0], npix))
            for num in range(alm_hi_jsim.shape[0]):
                    Mhi[num] = hp.alm2map(np.ascontiguousarray(alm_hi_jsim[num]), params_sims_.nside, pol=False, verbose=False)    
            #
            Xfg  = dcopy(Xfg0)
            Xsky = Xfg + f*Mhi;
            X    = Xsky #+ Mhi
            # --- Filtering
            params_cs, params_wt = cs.load(params_CS_, params_WT_)            
            X  = cs.adaptation_maps(X, params_maps_, params_inout_)
            Xr = cs.maps2CSmaps(X, params_wt, params_cs, verbose=cs_verbose)# , False)
            R  = Xr['reconstruction']['21cm']
            # -------------------------------------
            alm_g_sim = dict_all_[jsim]['alm_kappa_sim'][2:,:]#mock
            Xg        = hp.alm2map(alms=np.ascontiguousarray(alm_g_sim[0]),nside=256,pol=False)
            #
            cx_res = np.vstack([ hp.anafast(map1=iR, map2=Xg, pol=False) for iR in R   ])
            cx_inp = np.vstack([ hp.anafast(map1=iH, map2=Xg, pol=False) for iH in Mhi ])            
                
            alpha_cx = cx_inp/cx_res
            alpha_cx_list.append(alpha_cx.flatten())
                
            ndim_cx = alpha_cx.shape
            alpha_cx_flat = np.vstack(alpha_cx_list)
            alpha_cx_sim, alpha_cx_sim_err = statc.get_estimator(alpha_cx_flat, tf_estimator, axis=0)#, **kwargs)
            alpha_cx_sim_err = np.sqrt(alpha_cx_sim_err).reshape(N,-1)
            alpha_cx_sim = alpha_cx_sim.reshape(N,-1)
        return {'tf':alpha_cx_sim, 'errtf':alpha_cx_sim_err, 'realization_mocks': reali_mocks, 'realization_obs':reali_obs,
                'HIobs':Xhi0,'FGobs':Xfg0}      
    else:
        raise NameError('Type not accept: ', type_.lower())
        

In [6]:
for ii,gbin in enumerate([1,2,3,4,5,6,7]):
    timei       = time.time()
    params_general = pd.Series({'verbose'  : 1,
                                'project'  : 'CX',
                                'threshold': 1e-25,
                                'optimized': False,
                               })
    
    params_sims = pd.Series({'nreals'         : 100,
                             'nside'          : 256,                                          
                             'lmin'           : 0,                         
                             'lmax'           : int(3*256),
                             'apply_mask'     : 0,
                             'include_wn'     : 0,
                             'include_beam'   : 0,
                             'wn_sigma_pix'   : 0.154,
                             'beam_fwhm'      : 0.011635,        
                             'numin'          : 980,
                             'numax'          : 1260,
                             'nbands'         : 30,
                             'stokes'         : 'I',
                             'unit'           : 'mK',
                            })
                   
    params_tf = pd.Series({'tf_hixhi'        : 1,
                           'tf_gxhi'         : 1,
                           'auto_g_bin'      : gbin,
                           'cross_g_bin'     : gbin,
                           'auto_hi_bins'    : -1,
                           'cross_hi_bins'   : -1,
                           #'tf_variance'     : tf_variance, 
                           'estimator'       : 'huber',
                           'save_simulated'  : 1,
                           'save_theoretical': 1,
                           'save_hdf5'       : 1,
                           'save_txt'        : 0,
                          })   
    
    params_chisel = pd.Series({'method'    :'ICA',
                               'wtransform':'identity',
                               'ns'        : 4})            
    
    params_inout = pd.Series({'dirpath_chisel'      : '/data/AMARINS/chisel/scripts',
                              'dirpath_sims'        : '/data/AMARINS/LSSxHI-DATA/simulations/GCxHI/LSSTxLOWZ_40_0001/bin{b}',
                              'dirpath_foregrounds' : '/data/AMARINS/MAPS/FG256',
                              'dirpath_mask'        : '/data/AMARINS/MAPS/MASK',
                              #'dirpath_savedata'    : dirpath_savedata,                         
                              'filename_foregrounds': 'FG_I_256_980mhz1260mhz_30bins_full_nonfrps_L0.fits',                
                              #'filename_mask'       : filename_mask,
                              'filepath_field1'     : '/data/AMARINS/LSSxHI-DATA/theoretical/GCxHI/LSST_bin{b}_40_0001_HI_cl_LOWZ.txt',
                              'filepath_field2'     : '/data/AMARINS/LSSxHI-DATA/theoretical/GCxHI/LSST_bin{b}_40_0001_DENSITY_cl_LOWZ.txt',
                              'filepath_cross'      : '/data/AMARINS/LSSxHI-DATA/theoretical/GCxHI/LSST_bin{b}_40_0001_HIxDENSITY_cl_LOWZ.txt',
                              'pathout'             : '/data/AMARINS/LSSxHI-DATA/tfunction/LSST_40_0001__100sims_HIsky'
                           })
    
    #params_tf['auto_g_bin'] = gbin
    params_sims['l'] = np.arange(params_sims['lmin'], params_sims['lmax']+1, 1)	
    
    try:
        if params_tf['auto_hi_bins']<0:
            params_tf['auto_hi_bins'] = np.arange(params_sims['nbands'], dtype=np.int32)
    except:
        pass
    ################################
    try:    
        if params_tf['cross_hi_bins']<0:
        #    params_inout['output']['cross_hi_bins'] = np.arange(params_fields['hi']['nbands'], dtype=np.int32)
            if params_tf['auto_g_bin']==0:
                params_tf['cross_hi_bins'] = np.array([0,1,2,3])
            elif params_tf['auto_g_bin']==1:
                params_tf['cross_hi_bins'] = np.array([4,5,6,7,8,9])
            elif params_tf['auto_g_bin']==2:
                params_tf['cross_hi_bins'] = np.array([10,11,12,13,14])
            elif params_tf['auto_g_bin']==3:
                params_tf['cross_hi_bins'] = np.array([15,16,17,18])
            elif params_tf['auto_g_bin']==4:
                params_tf['cross_hi_bins'] = np.array([18,19,20,21])
            elif params_tf['auto_g_bin']==5:
                params_tf['cross_hi_bins'] = np.array([22,23,24])
            elif params_tf['cross_g_bin']==6:
                params_tf['cross_hi_bins'] = np.array([25,26,27])        
            elif params_tf['auto_g_bin']==7:
                params_tf['cross_hi_bins'] = np.array([28,29])       
            else:
                params_tf['cross_hi_bins'] = np.array([28,29])             
    except:
        pass
    params_tf['cross_g_bin'] = dcopy( np.asarray( params_tf['cross_g_bin'] ) )
    params_tf['auto_g_bin' ] = dcopy( np.asarray( params_tf['auto_g_bin' ] ) )
    
    try:
        params_general['project' ] = params_general['project' ].format(b=params_tf['auto_g_bin'])    
    except:
        pass
    try:
        params_inout['dirpath_savedata' ] = params_inout['dirpath_savedata' ].format(b=params_tf['auto_g_bin'])    
    except:
        pass         
    try:
        params_inout['dirpath_sims' ] = params_inout['dirpath_sims' ].format(b=params_tf['auto_g_bin'])    
    except:
        pass     
    try:    
        params_inout['filepath_field1'] = params_inout['filepath_field1'].format(b=params_tf['auto_g_bin'])    
    except:
        pass
    try:
        params_inout['filepath_field2'] = params_inout['filepath_field2'].format(b=params_tf['auto_g_bin'])    
    except:
        pass   
    try:    
        params_inout['filepath_cross' ] = params_inout['filepath_cross' ].format(b=params_tf['auto_g_bin'])    
    except:
        pass    
    ####
    params_maps    = loadparams.load_params_maps()
    params_CS      = loadparams.load_params_CS()
    params_WT      = loadparams.load_params_WT()
    ####
    del params_maps['iseed']    
    params_CS['method']     = dcopy(params_chisel['method'])
    params_CS['wtransform'] = dcopy(params_chisel['wtransform'])
    params_CS['ns']         = dcopy(params_chisel['ns'])
    ##############################################################################################################################
    Mfg = hdata.getmap(dirpath_=params_inout.dirpath_foregrounds, filename_=params_inout.filename_foregrounds, healpix_readingformat=False, hdu=1)
    if (params_sims.apply_mask) and (params_inout.dirpath_mask!="") and (params_inout.filename_mask!=""):
        MASK = hdata.getmap(dirpath_=params_inout.dirpath_mask, filename_=params_inout.filename_mask, healpix_readingformat=False, hdu=1)
        params_chisel['coverage_flag'] = 'masked'
        if params_inout['pathout'] =="":
            params_inout['pathout'] = 'masked'    
    else:
        MASK = np.ones_like(Mfg[0,:])
        params_chisel['coverage_flag'] = 'fullsky'
        if params_inout['pathout'] =="":
            params_inout['pathout'] = 'fullsky'        

    ##############################################################################################################################
    output_info = {'field':'field', 'nside':params_sims.nside, 
                   'frequency'   :{'min':params_sims.numin, 'max':params_sims.numax,'nbands':params_sims.nbands,  'unit':params_sims.unit}, 
                   'stokes'      :params_sims['stokes'],
                   'coverage'    :None,
                   'apply_beam'  :False, 
                   'namePL'      :False, 
                   'nameFG'      :params_inout.filename_foregrounds,               
                   'pathdir_N'   :False
                  }
    
    output_info['filename'] = hdata.new_formattingnames(F=output_info['field'], 
                                                        S=output_info['stokes'], 
                                                        NSIDE_=output_info['nside'], 
                                                        freq_min=int(output_info['frequency']['min']), 
                                                        freq_max=int(output_info['frequency']['max']),
                                                        freq_unit=output_info['frequency']['unit'],
                                                        Num_=output_info['frequency']['nbands'], 
                                                        C=output_info['coverage'], 
                                                        Bres=None,Bmodel=None,A=None, R='')
    ##############################################################################################################################
    clf1 = np.loadtxt(params_inout['filepath_field1']).T
    clf2 = np.loadtxt(params_inout['filepath_field2']).T
    clcx = np.loadtxt(params_inout['filepath_cross'] ).T 
    print('Theoretical loaded!')

    print('loading sims...')
    itime         = time.time() #['alm_hi_sim', 'cl_hi_sim', 'cl_kappa_sim', 'cl_cross_sim', 'alm_kappa_sim']
    varnames      = ['alm_hi_sim','alm_kappa_sim' , 'cl_hi_sim', 'cl_kappa_sim', 'cl_cross_sim']#,'cross_correlation_coef']
    dict_all      = cxfs.get_file_simulated_data(path_=params_inout['dirpath_sims'], varnames=varnames, nsims_to_use=params_sims.nreals)
    dict_all_mean = cxfs.dict_averages_from_loaded_data(dict_all_=dict_all, varnames_=varnames)
    nsims         = int(dict_all_mean[varnames[0]]['bin1']['matrix'].shape[0]    )
    if params_general['verbose']: print('Loading time for {0:d} sims: {1:.2f} seg'.format(nsims, time.time()-itime)) 
    del nsims    

    params_sims['realizations'] = np.asarray(list(dict_all.keys()))
    if params_general['verbose']: print("Using sims: ",params_sims['realizations'])
    N=params_sims['nbands']        
    print('Sims loaded!')        
    #######################################################################################################
    if params_general['verbose']: 
        print('\n------------------------------------------------------------------')
        print('Starting the Foreground Removal Process')
        print('And the Transfer Function estimation   ')
        print('GBIN: ', params_tf['cross_g_bin'])        
        print('HBIN: ', params_tf['cross_hi_bins'])                
        print('------------------------------------------------------------------\n')
    tf_hi = tf_processing(dict_all_=dict_all, 
                          reali_mocks=params_sims['realizations'][1:], 
                          reali_obs=params_sims['realizations'][0], 
                          params_sims_=params_sims, params_maps_=params_maps, params_inout_=params_inout,
                          tf_estimator=params_tf['estimator'],
                          params_WT_=params_WT,  params_CS_=params_CS, obs_include_hi=True,
                          verbose=True, verbose_level=0, type_='switzer')
    print('TF_HI: done!')
    tf_no = tf_processing(dict_all_=dict_all, 
                          reali_mocks=params_sims['realizations'][1:], 
                          reali_obs=params_sims['realizations'][0], 
                          params_sims_=params_sims, params_maps_=params_maps, params_inout_=params_inout,
                          tf_estimator=params_tf['estimator'],
                          params_WT_=params_WT,  params_CS_=params_CS, obs_include_hi=False,
                          verbose=True, verbose_level=1, type_='switzer')
    print('TF_noHI: done!')
    ####
    # OBS data
    Xfg0 = hdata.getmap(dirpath_=params_inout.dirpath_foregrounds, 
                   filename_=params_inout.filename_foregrounds, 
                   healpix_readingformat=False, 
                   hdu=1)
    jsim0      = params_sims['realizations'][0]
    alm_g_sim = dict_all[jsim0]['alm_kappa_sim'][2:,:]
    # --- Loading Xhi realization
    npix = hp.nside2npix(params_sims.nside)
    Xhi  = tf_hi['HIobs']  
    params_maps["getdata"] = "observed"
    # --- Filtering 1
    Xfg  = dcopy(Xfg0)
    Xsky = Xfg + Xhi;
    X    = Xsky
    #
    params_cs, params_wt = cs.load(params_CS, params_WT)
    X   = cs.adaptation_maps(X, params_maps, params_inout)
    Xr  = cs.maps2CSmaps(X, params_wt, params_cs)# , False)
    Ae  = Xr['mixmatrix']
    Xg  = hp.alm2map(alms=np.ascontiguousarray(alm_g_sim[0]),nside=256,pol=False)
    R   = Xr['reconstruction']['21cm']
    #
    CX = np.vstack([ hp.anafast(map1=iR,map2=Xg,pol=False) for iR in R])
    jsim0 = tf_hi['realization_obs']
    if not ii:
        TF_dict_hi = {gbin: {'tf_hi':tf_hi,'tf_no':tf_no,'jsim0':jsim0,'CX':CX,'R':R,'Xg':Xg,'Xr':Xr}}
    else:
        TF_dict_hi[gbin] = {'tf_hi':tf_hi,'tf_no':tf_no,'jsim0':jsim0,'CX':CX,'R':R,'Xg':Xg,'Xr':Xr}

    timef = dcopy( time.time() - timei )  
    print('\n------------------------------------------')
    print('END {} | running time: {} min'.format(ii, np.around(timef/60,decimals=2)))
    print('------------------------------------------\n')    
    del tf_hi, tf_no, CX, R, Xg, X, Xr, Ae, params_cs, params_wt, dict_all

Theoretical loaded!
loading sims...
Loading time for 100 sims: 407.04 seg
Using sims:  ['sim0' 'sim1' 'sim10' 'sim100' 'sim1000' 'sim1001' 'sim1002' 'sim1003'
 'sim1004' 'sim1005' 'sim1006' 'sim1007' 'sim1008' 'sim1009' 'sim101'
 'sim1010' 'sim1011' 'sim1012' 'sim1013' 'sim1014' 'sim1015' 'sim1016'
 'sim1017' 'sim1018' 'sim1019' 'sim102' 'sim1020' 'sim1021' 'sim1022'
 'sim1023' 'sim1024' 'sim1025' 'sim1026' 'sim1027' 'sim1028' 'sim1029'
 'sim103' 'sim1030' 'sim1031' 'sim1032' 'sim1033' 'sim1034' 'sim1035'
 'sim1036' 'sim1037' 'sim1038' 'sim1039' 'sim104' 'sim1040' 'sim1041'
 'sim1042' 'sim1043' 'sim1044' 'sim1045' 'sim1046' 'sim1047' 'sim1048'
 'sim1049' 'sim105' 'sim1050' 'sim1051' 'sim1052' 'sim1053' 'sim1054'
 'sim1055' 'sim1056' 'sim1057' 'sim1058' 'sim1059' 'sim106' 'sim1060'
 'sim1061' 'sim1062' 'sim1063' 'sim1064' 'sim1065' 'sim1066' 'sim1067'
 'sim1068' 'sim1069' 'sim107' 'sim1070' 'sim1071' 'sim1072' 'sim1073'
 'sim1074' 'sim1075' 'sim1076' 'sim1077' 'sim1078' 'sim1079' 'sim10

In [ ]:
gbin=1
TF_dict_hi

In [7]:
fig   = plt.figure()
grid  = plt.GridSpec(1,1,top=1.0,right=1.8, wspace=0.03, hspace=0.35)
############################################################
ax  = plt.subplot(grid[0]) 
#
index_bin = 3
ibin = params_tf['cross_hi_bins'][index_bin]
#
l = np.arange(3*256)#leff[filter_ell]
fact = 1#l*(l+1)/2/np.pi
clrec = b.bin_cell( fact*CX[ibin,:] )
cldeb_no = b.bin_cell( fact*CX[ibin,:]*TF_no['tf'][ibin,:] )
cldeb_hi = b.bin_cell( fact*CX[ibin,:]*TF_hi['tf'][ibin,:] )
clthr = dict_all[jsim0]['cl_cross_sim'][1:]
clthr = b.bin_cell(  fact*clthr[ibin,:] )
#
ax.plot(leff, clthr, color='black', lw=1)
ax.scatter(leff, clrec, color='blue', s=10)
ax.scatter(leff, cldeb_no, color='red', s=100, marker='x')
ax.scatter(leff, cldeb_hi, color='forestgreen', s=100, marker='+')
for l in leff: ax.axvline(x=l, c='black', ls='dashed', lw=0.3)
ax.set_xlim(30,400)  
#ax.set_ylim(1e-7,3e-5)  
ax.set_yscale('log'); ax.set_xscale('linear')
############################################
plt.show()

1


In [11]:
#TF_dict_hi
import joblib
joblib.dump(TF_dict_hi, "tf_data.joblib", compress=3)

['tf_data.joblib']

In [13]:
import pickle
with open("tf_data.pkl", "wb") as f:
    pickle.dump(TF_dict_hi, f, protocol=pickle.HIGHEST_PROTOCOL)